# Face Anonymization from 5 Given Landmarks

Geometric face removal for Einstar photogrammetry scans. Given the 5
landmarks (Nz, Iz, Cz, Lpa, Rpa), `anonymize_scan` deletes the face and
returns the result in the raw Einstar (`crs="digitized"`) frame, so the
saved `.obj` lines up with the original at co-registration.

Flow: load &rarr; pick 5 landmarks &rarr; `anonymize_scan` &rarr; save.

By default this notebook operates on the example scan provided by
`cedalion.data.get_photogrammetry_example_scan()`. Set `fname_scan` below to
run it on your own scan.

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
import logging
import os
import tempfile
from pathlib import Path

import numpy as np
import pyvista as pv
import trimesh
import xarray as xr
from PIL import Image

import cedalion
import cedalion.data
import cedalion.dataclasses as cdc
import cedalion.io
import cedalion.vis.blocks as vbx
from cedalion.geometry.landmarks import normalize_landmarks_labels
from cedalion.geometry.photogrammetry.anonymization import (
    CapDetectionParams,
    anonymize_scan,
    save_anonymized_scan,
)
from cedalion.vtktutils import trimesh_to_vtk_polydata

xr.set_options(display_expand_data=False)

logging.basicConfig()
logging.getLogger("cedalion").setLevel(logging.WARNING)
logging.getLogger("trame_client").setLevel(logging.WARNING)
logging.getLogger("trame_server").setLevel(logging.WARNING)

## 0. Choose between interactive and static mode

This example notebook provides two modes, controlled by the constant `INTERACTIVE`:
- a static mode intended for rendering the documentation. It uses the
  hand-picked landmarks of the example scan.
- an interactive mode, in which the 3D visualizations react to user input. The
  landmark picking needs these interactive plots.

In [ ]:
INTERACTIVE = False

if INTERACTIVE:
    # option 1: render in the browser
    # pv.set_jupyter_backend("client")
    # option 2: offload rendering to a server process using trame
    pv.set_jupyter_backend("server")
else:
    pv.set_jupyter_backend("static")  # static rendering (for documentation page)

## 1. Load the Einstar scan

By default the example scan is used. Set `fname_scan` to the path of your own
`.obj` to anonymize that instead. The texture is expected as a sibling `.jpg`;
keep it next to the `.obj`.

`scan_units` declares the units of the vertex coordinates *in the file*: Einstar
exports millimetres, other scanners (e.g. Scaniverse) export metres. The
anonymization works in mm - `anonymize_scan`'s `*_mm` arguments and
`CapDetectionParams` are plain mm floats, and the interactive landmark picker
tags picked points as mm - so a non-mm scan is rescaled here and scaled back to
`scan_units` before saving.

In [ ]:
# insert here your own file if you do not want to use the example
fname_scan = ""  # path to .obj scan file
scan_units = cedalion.units.m  # units of the vertex coordinates in the file (Scaniverse is typically in m, Einstar in mm)

USING_EXAMPLE_SCAN = not fname_scan

if USING_EXAMPLE_SCAN:
    fname_scan, fname_snirf, fname_montage_img = (
        cedalion.data.get_photogrammetry_example_scan()
    )
    scan_units = cedalion.units.mm  # the example scan is an Einstar scan, i.e. mm
    # the example scan lives in a read-only download cache -> write elsewhere
    out_dir = tempfile.gettempdir()
else:
    out_dir = str(Path(fname_scan).parent)

fname_out = str(Path(out_dir) / (Path(fname_scan).stem + "_anon.obj"))

surface = cedalion.io.read_einstar_obj(fname_scan, units=scan_units)

# Work in mm from here on. `surface.units` is a pint.Unit, which has no .to();
# multiply by 1 to get a Quantity first. Keep this right after loading: the
# in-place update is only safe while no KD-tree has been cached yet.
if surface.units != cedalion.units.mm:
    factor = (1 * surface.units).to(cedalion.units.mm).magnitude
    surface.mesh.apply_scale(factor)
    surface.units = cedalion.units.mm

print(f"loaded {fname_scan}")
print(f"{surface.nvertices:,} vertices, {surface.nfaces:,} faces")
print(f"anonymized scan will be written to {fname_out}")

In [ ]:
# trimesh 4.6 places the texture image on visual.material.image; if neither
# path has it, attach it from the sibling JPG so save_anonymized_scan can
# sanitize the final texture.
visual = surface.mesh.visual
img = getattr(visual, "image", None) or getattr(
    getattr(visual, "material", None), "image", None
)

if img is None:
    fname_jpg = str(Path(fname_scan).with_suffix(".jpg"))
    uv = getattr(visual, "uv", None)
    assert os.path.exists(fname_jpg) and uv is not None, "no texture available"
    surface.mesh.visual = trimesh.visual.TextureVisuals(
        uv=uv,
        image=Image.open(fname_jpg).convert("RGBA"),
    )

## 2. Mask parameters

These control the extent of the anonymized region. The defaults work for the
example scan and are a reasonable starting point for other Einstar scans.

In [ ]:
EAR_DELETE_RADIUS_MM = 40.0      # sphere radius around LPA/RPA included in deletion
LANDMARK_KEEP_RADIUS_MM = 8.0    # sphere radius around each landmark kept intact
EYEBROW_OFFSET_MM = 10.0         # failsafe cap height above Nz (flush-cap fallback)
CAP_Z_CEILING_MM = 40.0          # mm above Nz where cap detection is clamped
HEAD_ISOLATION_RADIUS_MM = 220.0 # sphere radius used to strip body/shoulders

## 3. Pick the 5 landmarks

In interactive mode, right-click on the mesh to place a sphere; click a sphere
to cycle its label through `Nz -> Iz -> Cz -> Lpa -> Rpa`. Close the window when
all 5 are placed. This cell does nothing when `INTERACTIVE = False`.

In [ ]:
if INTERACTIVE:
    pvplt = pv.Plotter()
    get_landmarks = vbx.plot_surface(
        pvplt, surface, opacity=1.0, pick_landmarks=True
    )
    pvplt.show()

### Wrap the picked points into a `LabeledPoints` array

In static mode the hand-picked landmarks of the example scan are used. When
anonymizing your own scan without the picker, provide the landmarks in a TSV
file (see `cedalion.io.load_tsv`) and set `CACHED_LANDMARKS_TSV`.

In [ ]:
CACHED_LANDMARKS_TSV = None  # used when INTERACTIVE = False on your own scan

if INTERACTIVE:
    landmarks = get_landmarks()
elif CACHED_LANDMARKS_TSV:
    landmarks = cedalion.io.load_tsv(CACHED_LANDMARKS_TSV, crs="digitized", units="mm")
else:
    # For documentation purposes and to enable automatically rendered example
    # notebooks we provide the hand-picked coordinates of the example scan here.
    assert USING_EXAMPLE_SCAN, "set CACHED_LANDMARKS_TSV or INTERACTIVE = True"

    landmark_labels = ["Nz", "Iz", "Cz", "Lpa", "Rpa"]
    landmark_coordinates = np.asarray(
        [
            [-145.7705448870955, 238.9471323710179, 114.5570981799518],
            [-149.5434014519131, 208.5145848871951, 305.3606241041927],
            [-149.8162350654902, 329.0819859022904, 219.2744781036182],
            [-220.3323233580926, 202.9298496447481, 188.0019479653261],
            [-79.66694393192844, 201.6810162117443, 185.0682204348718],
        ]
    )

    # same constructor as load_tsv and as the interactive picker
    landmarks = cdc.build_labeled_points(
        landmark_coordinates,
        crs=surface.crs,
        units=surface.units,
        labels=landmark_labels,
        types=[cdc.PointType.LANDMARK] * len(landmark_labels),
    )

landmarks = normalize_landmarks_labels(landmarks)
labels = list(landmarks["label"].values)
assert set(labels) == {"Nz", "Iz", "Cz", "LPA", "RPA"}, f"bad labels: {labels}"
display(landmarks)

## 4. Anonymize

`anonymize_scan` isolates the head, aligns it to the CTF coordinate system,
detects the front edge of the cap and deletes the vertices below it. The
returned surface and landmarks are back in the `digitized` frame.

In [ ]:
surface_anon, landmarks_anon = anonymize_scan(
    surface,
    landmarks,
    head_isolation_radius_mm=HEAD_ISOLATION_RADIUS_MM,
    ear_delete_radius_mm=EAR_DELETE_RADIUS_MM,
    landmark_keep_radius_mm=LANDMARK_KEEP_RADIUS_MM,
    cap=CapDetectionParams(
        eyebrow_offset=EYEBROW_OFFSET_MM,
        z_ceiling=CAP_Z_CEILING_MM,
    ),
)

n_removed = surface.nvertices - surface_anon.nvertices
print(
    f"{Path(fname_scan).stem}: {surface.nvertices:,} -> "
    f"{surface_anon.nvertices:,} vertices (-{n_removed:,})"
)

## 5. Compare original and anonymized scan

The anonymization only deletes vertices, it does not move the remaining ones:
both meshes live in the same `digitized` frame and the surviving vertices keep
their original coordinates.

To make the two directly comparable, both subplots use the same explicit
camera in a frontal view. Its orientation is derived from the landmarks: the
camera looks from `Nz` towards the `LPA`/`RPA` midpoint, with the `Cz`
direction pointing up. Setting `camera_position` after the meshes were added -
together with `reset_camera=False` - prevents PyVista from framing each subplot
to its own mesh bounds. The remaining difference between the two panels is the
deleted face.

In [ ]:
def unit(v):
    return v / np.linalg.norm(v)


lm_colors = {"Nz": "lime", "Iz": "magenta", "Cz": "cyan", "LPA": "orange", "RPA": "blue"}

lm_pos = landmarks_anon.pint.to("mm").pint.dequantify().values
lm_lbls = landmarks_anon["label"].values
lm = {lbl: pos for lbl, pos in zip(lm_lbls, lm_pos)}

# look at the face from the front: from Nz towards the LPA-RPA midpoint, with
# the Cz direction pointing up.
center = 0.5 * (lm["LPA"] + lm["RPA"])
front = unit(lm["Nz"] - center)
up = unit(lm["Cz"] - center)
camera = (tuple(center + 700 * front), tuple(center), tuple(up))

In [ ]:
pvplt = pv.Plotter(shape=(1, 2), window_size=(1200, 600))
pvplt.link_views()

for i, (surf, title) in enumerate(
    [
        (surface, "Original"),
        (surface_anon, f"Anonymized (-{n_removed:,} verts)"),
    ]
):
    pvplt.subplot(0, i)
    pvplt.add_mesh(
        pv.wrap(trimesh_to_vtk_polydata(surf.mesh)),
        rgb=True,
        smooth_shading=True,
        reset_camera=False,
    )
    for lbl, pos in lm.items():
        pvplt.add_mesh(
            pv.Sphere(radius=4, center=pos),
            color=lm_colors[lbl],
            reset_camera=False,
        )
    pvplt.add_text(title, position="upper_left", font_size=12)
    pvplt.camera_position = camera

pvplt.show()

Overlaying both meshes confirms that they are aligned: the anonymized surface
(red) lies exactly on the original one (transparent white), which sticks out
only where the face was removed.

In [ ]:
pvplt = pv.Plotter(window_size=(700, 700))
pvplt.add_mesh(
    pv.wrap(trimesh_to_vtk_polydata(surface.mesh)),
    color="w",
    opacity=0.35,
    reset_camera=False,
)
pvplt.add_mesh(
    pv.wrap(trimesh_to_vtk_polydata(surface_anon.mesh)),
    color="red",
    reset_camera=False,
)
pvplt.camera_position = camera
pvplt.show()

## 6. Save

The mesh is scaled back to `scan_units` first, so the written file has the same
scale as the input scan and overlays it directly.

For `.obj` output `save_anonymized_scan` writes the mesh, an `.mtl` and a
sanitized `.jpg`: the texture is rebuilt so that the face region contains no
image data anymore.

In [ ]:
# The pipeline worked in mm; write the result in the input file's units so the
# anonymized .obj is a drop-in replacement for the original scan.
if surface_anon.units != scan_units:
    factor = (1 * surface_anon.units).to(scan_units).magnitude
    surface_anon.mesh.apply_scale(factor)
    surface_anon.units = scan_units
    landmarks_anon = landmarks_anon.pint.to(scan_units)

written = save_anonymized_scan(surface_anon, fname_out)
for p in written:
    print(f"wrote {p}")